# Gerador de PDFs Premium (Medhelp)
Este notebook varre a pasta `Resumos_Prontos`, lê os arquivos Markdown puros gerados pelo Apps Script, converte para HTML e aplica um CSS avançado (nível editora) usando a biblioteca `WeasyPrint` para gerar PDFs lindíssimos.

**Novidade:** Ele foi configurado para **pular automaticamente** os resumos que já tiverem um PDF gerado na pasta final. Assim, sua pasta `Resumos_Prontos` pode continuar servindo de acervo sem causar reprocessamentos desnecessários!

In [ ]:
!pip install -q weasyprint markdown
!apt-get install -y -q pango1.0-tools

In [ ]:
import os
import markdown
from weasyprint import HTML, CSS
from google.colab import drive

# Conectar Google Drive
drive.mount("/content/drive")

RESUMOS_DIR = "/content/drive/MyDrive/Logística - Drive/Transcrições/Resumos_Prontos"
PDFS_DIR = "/content/drive/MyDrive/Logística - Drive/Transcrições/PDFs_Premium"

os.makedirs(PDFS_DIR, exist_ok=True)

In [ ]:
CSS_PREMIUM = """
@import url("https://fonts.googleapis.com/css2?family=Montserrat:wght@400;600;700;800&family=Inter:wght@400;500;600&display=swap");

@page {
    size: A4;
    margin: 25mm 30mm 25mm 25mm;
    @bottom-center {
        content: counter(page);
        font-family: "Inter", sans-serif;
        font-size: 10pt;
        color: #94a3b8;
    }
    @bottom-left {
        content: "© Conteúdo Autoral • João Gabriel R. Trovão";
        font-family: "Inter", sans-serif;
        font-size: 9pt;
        color: #64748b;
    }
    @top-right {
        content: "Material Estruturado de Revisão Clínica • Medhelp";
        font-family: "Montserrat", sans-serif;
        font-size: 8pt;
        color: #cbd5e1;
        text-transform: uppercase;
        letter-spacing: 1px;
    }
}

body {
    font-family: "Inter", sans-serif;
    font-size: 11pt;
    line-height: 1.6;
    color: #333333;
    background-color: #ffffff;
}

h1, h2, h3, h4 {
    font-family: "Montserrat", sans-serif;
    color: #111827;
    margin-top: 1.5em;
    margin-bottom: 0.5em;
    page-break-after: avoid;
}

h1 {
    font-size: 18pt;
    font-weight: 800;
    color: #000000;
    letter-spacing: -0.5px;
    padding-bottom: 6pt;
    margin-top: 0;
}

h2 {
    font-size: 14pt;
    font-weight: 700;
    color: #000000;
    padding: 8pt 0pt;
}

h3 {
    font-size: 12pt;
    font-weight: 600;
    color: #333333;
}

strong {
    color: #111827;
    font-weight: 600;
}

blockquote {
    border-left: 4pt solid #cbd5e1;
    margin: 1.5em 0;
    padding: 10pt 14pt;
    background-color: #f8fafc;
    color: #475569;
    font-style: italic;
    border-radius: 0 8pt 8pt 0;
    page-break-inside: avoid;
}

table {
    width: 100%;
    border-collapse: collapse;
    margin: 2em 0;
    font-size: 10pt;
    page-break-inside: auto;
}

tr {
    page-break-inside: avoid;
    page-break-after: auto;
}

th {
    background-color: #f8fafc;
    color: #000000;
    padding: 10pt 12pt;
    text-align: left;
    font-family: "Inter", sans-serif;
    font-weight: 700;
    border-bottom: 2pt solid #e2e8f0;
    border-top: 1pt solid #e2e8f0;
}

td {
    padding: 10pt 12pt;
    border-bottom: 1pt solid #e2e8f0;
    background-color: #ffffff;
    vertical-align: top;
}

ul, ol {
    padding-left: 20pt;
}

li {
    margin-bottom: 6pt;
}
"""

processados = 0
ignorados = 0
for filename in os.listdir(RESUMOS_DIR):
    if filename.endswith(".md"):
        # Verifica se o PDF já existe antes de fazer qualquer coisa
        pdf_filename = filename.replace(".md", ".pdf")
        pdf_path = os.path.join(PDFS_DIR, pdf_filename)
        
        if os.path.exists(pdf_path):
            ignorados += 1
            continue
            
        filepath = os.path.join(RESUMOS_DIR, filename)
        
        with open(filepath, "r", encoding="utf-8") as f:
            md_content = f.read()
            
        html_body = markdown.markdown(md_content, extensions=["tables", "fenced_code"])
        
        final_html = f"""
        <!DOCTYPE html>
        <html lang="pt-BR">
        <head>
            <meta charset="utf-8">
        </head>
        <body>
            {html_body}
        </body>
        </html>
        """
        
        # Gera o PDF via WeasyPrint
        HTML(string=final_html).write_pdf(
            pdf_path,
            stylesheets=[CSS(string=CSS_PREMIUM)]
        )
        print(f"✅ PDF gerado: {pdf_filename}")
        processados += 1

if processados == 0:
    print(f"Nenhum arquivo .md NOVO encontrado. ({ignorados} arquivos antigos foram ignorados).")
else:
    print(f"\n🎉 Sucesso! {processados} PDFs gerados com qualidade premium. ({ignorados} antigos ignorados).")